**TLDR**: Turns the Phase 1 exploration code into a reusable module, `src/porkqc/data.py`, which loads each COCO split, labels every image PASS (0) or FAIL (1), and raises clear errors for missing files or broken annotations. Eight pytest tests check the labels and the error handling on a small synthetic dataset. Work is done on the `feature/data-loader` branch and merged into `main` through a pull request.

#Set-up



In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection"
ZIP_PATH = "/content/drive/MyDrive/MMA3001/Pork Rasher Error-Packaging-.v4i.coco.zip"
%cd "{REPO_DIR}"

token = userdata.get("GITHUB_TOKEN")
PUSH_URL = f"https://{token}@github.com/chrixxv1/Pork-Tray-Inspection.git"

!mkdir -p /content/data
!unzip -oq "{ZIP_PATH}" -d /content/data
DATA_DIR = "/content/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection


##Branching

In [3]:
!git switch main
!git switch -c feature/data-loader

M	notebooks/0_setup.ipynb
M	notebooks/1_exploring_data.ipynb
Already on 'main'
Your branch is ahead of 'origin/main' by 7 commits.
  (use "git push" to publish your local commits)
Switched to a new branch 'feature/data-loader'


## Package Configuration



In [10]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "porkqc"
version = "0.1.0"
description = "Tray-level pass/fail inspection of packaged pork rashers"
requires-python = ">=3.10"
dependencies = ["numpy", "pillow"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]

Writing pyproject.toml


In [11]:
%%writefile src/porkqc/__init__.py
"""porkqc - tray-level pass/fail inspection of packaged pork rashers."""
__version__ = "0.1.0"

Writing src/porkqc/__init__.py


## Data Module

In [12]:
%%writefile src/porkqc/data.py
"""Loading of the Roboflow COCO export and definition of the tray-level label.

The dataset provides bounding boxes for five defect classes. This project
reduces them to a single image-level (tray-level) label:

* ``1`` (FAIL) - the image contains at least one defect annotation;
* ``0`` (PASS) - the image contains no defect annotation.

This reduction is a deliberate scoping decision: the validation and test splits
contain only 3-4 instances of the rarer classes, which is too few to evaluate
per-class detection, but the binary label has 200 evaluation images.
"""

from __future__ import annotations

import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

#: Splits shipped in the Roboflow export.
SPLITS = ("train", "valid", "test")

#: Expected image size (width, height) after Roboflow pre-processing.
EXPECTED_SIZE = (720, 540)


@dataclass(frozen=True)
class ImageRecord:
    """One image and its tray-level label.

    Attributes:
        path: Absolute path to the JPEG file.
        source_name: Name of the original (pre-augmentation) frame. Augmented
            copies of the same frame share this name.
        label: 1 if the image contains any defect annotation, else 0.
        categories: Sorted names of the defect classes annotated in the image.
    """

    path: Path
    source_name: str
    label: int
    categories: tuple[str, ...]


def load_split(split_dir: str | Path) -> list[ImageRecord]:
    """Load one split of the COCO export.

    Args:
        split_dir: Directory containing ``_annotations.coco.json`` and the
            images it references.

    Returns:
        One :class:`ImageRecord` per image, in the order listed in the JSON.

    Raises:
        FileNotFoundError: If the annotation file or a referenced image is
            missing.
        ValueError: If the annotation file is not valid COCO (missing keys,
            annotations referring to unknown images or categories).
    """
    split_dir = Path(split_dir)
    ann_path = split_dir / "_annotations.coco.json"
    if not ann_path.is_file():
        raise FileNotFoundError(f"No COCO annotation file at {ann_path}")

    with ann_path.open() as f:
        coco = json.load(f)
    for key in ("images", "annotations", "categories"):
        if key not in coco:
            raise ValueError(f"{ann_path} is missing the '{key}' key")

    cat_names = {c["id"]: c["name"] for c in coco["categories"]}
    image_ids = {im["id"] for im in coco["images"]}

    cats_per_image: dict[int, set[str]] = {i: set() for i in image_ids}
    for ann in coco["annotations"]:
        if ann["image_id"] not in image_ids:
            raise ValueError(f"Annotation {ann['id']} refers to unknown image {ann['image_id']}")
        if ann["category_id"] not in cat_names:
            raise ValueError(f"Annotation {ann['id']} has unknown category {ann['category_id']}")
        cats_per_image[ann["image_id"]].add(cat_names[ann["category_id"]])

    records = []
    for im in coco["images"]:
        path = (split_dir / im["file_name"]).resolve()
        if not path.is_file():
            raise FileNotFoundError(f"Image listed in {ann_path} not found: {path}")
        cats = tuple(sorted(cats_per_image[im["id"]]))
        # Roboflow stores the pre-augmentation file name in extra.name; fall
        # back to file_name for exports that do not include it.
        source = im.get("extra", {}).get("name", im["file_name"])
        records.append(ImageRecord(path, source, int(bool(cats)), cats))
    return records


def load_dataset(root: str | Path) -> dict[str, list[ImageRecord]]:
    """Load the train, valid and test splits from the export root directory.

    Args:
        root: Directory containing ``train/``, ``valid/`` and ``test/``.

    Returns:
        Mapping from split name to its list of records.
    """
    root = Path(root)
    return {s: load_split(root / s) for s in SPLITS}


def summarise(records: list[ImageRecord]) -> dict[str, object]:
    """Summarise a split for reporting and sanity checks.

    Args:
        records: Records of one split.

    Returns:
        Dictionary with the number of images, number of unique source frames,
        fraction labelled FAIL and the number of images containing each class.
    """
    n = len(records)
    class_counts = Counter(c for r in records for c in r.categories)
    return {
        "n_images": n,
        "n_source_frames": len({r.source_name for r in records}),
        "fail_fraction": sum(r.label for r in records) / n if n else float("nan"),
        "images_per_class": dict(sorted(class_counts.items())),
    }

Writing src/porkqc/data.py


##Installing Package

In [2]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for porkqc (pyproject.toml) ... done


## Trying On Real Data

In [3]:
from porkqc.data import load_dataset, summarise

ds = load_dataset(DATA_DIR)
for split, records in ds.items():
    print(split, summarise(records))
print(ds["train"][0])

train {'n_images': 3349, 'n_source_frames': 1085, 'fail_fraction': 0.795759928336817, 'images_per_class': {'loose-meat': 66, 'packaging-error': 1541, 'twisted-meat': 69, 'unsealed': 1055, 'wrinkle': 18}}
valid {'n_images': 120, 'n_source_frames': 120, 'fail_fraction': 0.8833333333333333, 'images_per_class': {'loose-meat': 3, 'packaging-error': 23, 'twisted-meat': 3, 'unsealed': 81}}
test {'n_images': 80, 'n_source_frames': 80, 'fail_fraction': 0.85, 'images_per_class': {'loose-meat': 2, 'packaging-error': 16, 'twisted-meat': 4, 'unsealed': 49}}
ImageRecord(path=PosixPath('/content/data/train/frame_with_red_8214_jpg.rf.5acda82172bcf022bd7f017eb084474d.jpg'), source_name='frame_with_red_8214.jpg', label=1, categories=('packaging-error',))


## Test Fixture

In [4]:
%%writefile tests/conftest.py
"""Shared pytest fixtures."""

import json

import pytest
from PIL import Image


@pytest.fixture
def tiny_coco(tmp_path):
    """Write a 3-image COCO split (one PASS, two FAIL) and return its folder.

    Image 0 has no annotations (PASS), image 1 has one 'unsealed' box and
    image 2 has an 'unsealed' and a 'wrinkle' box (both FAIL).
    """
    images = []
    for i in range(3):
        name = f"frame_{i}.jpg"
        Image.new("RGB", (720, 540), (40, 40, 40)).save(tmp_path / name)
        images.append({"id": i, "file_name": name, "height": 540, "width": 720,
                       "extra": {"name": f"src_{i}.jpg"}})
    coco = {
        "images": images,
        "categories": [{"id": 4, "name": "unsealed"}, {"id": 5, "name": "wrinkle"}],
        "annotations": [
            {"id": 1, "image_id": 1, "category_id": 4, "bbox": [0, 0, 10, 10]},
            {"id": 2, "image_id": 2, "category_id": 4, "bbox": [5, 5, 10, 10]},
            {"id": 3, "image_id": 2, "category_id": 5, "bbox": [50, 50, 10, 10]},
        ],
    }
    (tmp_path / "_annotations.coco.json").write_text(json.dumps(coco))
    return tmp_path

Writing tests/conftest.py


## Testing

In [5]:
%%writefile tests/test_data.py
"""Tests of COCO loading, label definition and input validation."""

import json

import pytest

from porkqc.data import load_split, summarise


def test_labels_follow_annotations(tiny_coco):
    records = load_split(tiny_coco)
    assert [r.label for r in records] == [0, 1, 1]
    assert records[0].categories == ()
    assert records[2].categories == ("unsealed", "wrinkle")


def test_source_name_comes_from_extra(tiny_coco):
    records = load_split(tiny_coco)
    assert records[1].source_name == "src_1.jpg"


def test_summary(tiny_coco):
    s = summarise(load_split(tiny_coco))
    assert s["n_images"] == 3
    assert s["fail_fraction"] == pytest.approx(2 / 3)
    assert s["images_per_class"] == {"unsealed": 2, "wrinkle": 1}


def test_missing_annotation_file(tmp_path):
    with pytest.raises(FileNotFoundError):
        load_split(tmp_path)


def test_missing_image_is_reported(tiny_coco):
    (tiny_coco / "frame_0.jpg").unlink()
    with pytest.raises(FileNotFoundError, match="frame_0"):
        load_split(tiny_coco)


def _edit_coco(folder, change):
    """Load the fixture's JSON, apply `change` to it, and save it back."""
    path = folder / "_annotations.coco.json"
    coco = json.loads(path.read_text())
    change(coco)
    path.write_text(json.dumps(coco))


def test_unknown_category_is_reported(tiny_coco):
    _edit_coco(tiny_coco, lambda c: c["annotations"][0].update(category_id=99))
    with pytest.raises(ValueError, match="unknown category"):
        load_split(tiny_coco)


def test_unknown_image_is_reported(tiny_coco):
    _edit_coco(tiny_coco, lambda c: c["annotations"][0].update(image_id=99))
    with pytest.raises(ValueError, match="unknown image"):
        load_split(tiny_coco)


def test_missing_key_is_reported(tiny_coco):
    _edit_coco(tiny_coco, lambda c: c.pop("categories"))
    with pytest.raises(ValueError, match="categories"):
        load_split(tiny_coco)

Writing tests/test_data.py


## Run Tests

In [6]:
!pytest -q

........                                                                 [100%]
8 passed in 0.47s


## AI Usage Log

In [8]:
%%writefile -a AI_USE_LOG.md
| 2026-10-25 | Claude | Explained how to write the code for the package configuration, src/porkqc/data.py (COCO loader, PASS/FAIL labels, input validation) and the 8 pytest tests | Ran the loader on the real dataset and checked its counts against my Phase 1 tables; ran the tests (8 passed); asked Claude to explain parts I did not understand |

Appending to AI_USE_LOG.md


## Commit

In [ ]:
!git add pyproject.toml src/porkqc/__init__.py src/porkqc/data.py tests/conftest.py tests/test_data.py notebooks/2_data_module.ipynb AI_USE_LOG.md
!git commit -m "Add data loader with tray-level labels, input validation and tests"
!git push {PUSH_URL} feature/data-loader

In [ ]:
!git switch main
!git pull {PUSH_URL} main
!git log --oneline -5